<div align="center">

# **Tesis: Conectividad Estructural de Tractos Témporo-Límbicos en Epilepsia**
### Archivo 3 de 3 — Modelado: CNN sobre Imágenes TDI y Benchmarking de Machine Learning Clásico

</div>

Este notebook es la **siguiente etapa** del pipeline, y consume directamente los artefactos que
generan los dos notebooks anteriores:

- **Del Archivo 1** (`features_por_sujeto.csv`): la matriz tabular de métricas escalares y
  geométricas, una fila por sujeto.
- **Del Archivo 2** (`tdi_grises/*_gris.png`): los cortes TDI en escala de grises normalizada, por
  clúster y por sujeto (Sección 6 del Archivo 2).

Implementa **dos enfoques de clasificación independientes** (epilepsia vs. control), para
comparar su desempeño:

1. **Sección A — CNN sobre imágenes TDI:** aprende patrones discriminativos directamente de la
   geometría espacial 2D de los tractos, sin reducirla primero a un puñado de métricas escalares.
2. **Sección B — Benchmarking de 5 clasificadores clásicos** sobre las métricas tabulares ya
   extraídas (FA, MD, AD, RD, volumen, geometría), con selección recursiva de características
   (RFECV).

> ⚠️ **Corrección respecto a un borrador previo de este notebook:** una versión anterior entrenaba
> los modelos con **etiquetas generadas aleatoriamente** (`np.random.randint` o reglas arbitrarias
> sobre el nombre del sujeto, sin relación con un diagnóstico real) — el *accuracy* resultante de
> eso no mide nada clínicamente válido, por más alto que salga el número. Esta versión **siempre**
> deriva `grupo_dx` de un manifiesto explícito (Sección 0.2) y detiene la ejecución si una etiqueta no es válida y, solo cuando no hay datos
> reales disponibles, genera un conjunto sintético **explícitamente etiquetado como tal**, con una
> señal estadística conocida y controlada (no ruido puro), únicamente para verificar que el pipeline
> corre de principio a fin.


> ### Versión cohorte (v2): qué cambia respecto a la versión anterior
> El **Archivo 2 no se modifica**: se ejecuta tal cual, una vez por sujeto, y este notebook consume sus salidas.
>
> - **Sección 0.2 (nueva):** ingesta por sujeto (un `.zip` por paciente o control) para una cohorte de aproximadamente **70 pacientes y 30 controles**, con un **manifiesto** (`manifiesto_cohorte.csv`) como única fuente de la etiqueta diagnóstica y de las covariables (edad, sexo, lateralidad del foco). Valida conteos, duplicados y archivos faltantes antes de modelar.
> - **Etiquetas:** `grupo_dx` ya no se infiere del nombre de la carpeta. En la versión anterior una carpeta sin patrón reconocido quedaba como "desconocido" y la celda A.4 la convertía en control sin avisar. Ahora una etiqueta inválida detiene la ejecución.
> - **Nombres de las imágenes TDI:** se leen con la etiqueta anatómica `tracto_<código>_<lado>_cluster<NN>` que genera el Archivo 2.
> - **Métricas geométricas del Archivo 2** (eigenvalores, elongación, esfericidad, tortuosidad, índices de lateralidad) se agregan a la tabla de la Sección B.
> - **Evaluación para 70:30:** con esa proporción un clasificador que siempre dice "epilepsia" acierta el 70 %, así que la exactitud no sirve como métrica principal. Se reportan AUC, exactitud balanceada y MCC; las particiones son estratificadas y agrupadas por sujeto; la selección del mejor modelo clásico se hace por validación cruzada repetida y no sobre el conjunto de prueba; y se agregan intervalos de confianza por bootstrap de sujetos.
> - **Sección B independiente:** ahora importa las métricas que usa (antes dependían de una celda de la Sección A).

# **0. Configuración del entorno**

In [ ]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=0.85)
pd.set_option("display.max_colwidth", 120)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CARPETA_SALIDA = "/content/" if IN_COLAB else "./"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local / Jupyter'}")
print(f"Carpeta de salida: {CARPETA_SALIDA}")


## **0.1. Ubicación de los artefactos de los Archivos 1 y 2**
Ajusta estas dos rutas a donde tengas, respectivamente, el `features_por_sujeto.csv` exportado por
el Archivo 1, y la carpeta raíz que contiene una subcarpeta por sujeto con sus `tdi_grises/*.png`
exportados por el Archivo 2 (ver la convención de nombre de subcarpeta en la Sección 1.2).

In [ ]:
RUTA_FEATURES_TABULARES = CARPETA_SALIDA + "features_por_sujeto.csv"
CARPETA_TDI_COHORTE = CARPETA_SALIDA + "tdi_cohorte/"
os.makedirs(CARPETA_TDI_COHORTE, exist_ok=True)

if IN_COLAB:
    print("Si ya tenés 'features_por_sujeto.csv' (Archivo 1) y/o la carpeta de TDI por sujeto "
          "(Archivo 2), subilos ahora a este entorno (p. ej. arrastrándolos al panel de archivos "
          "de la izquierda, o montando Google Drive) y ajustá las dos rutas de arriba si hace "
          "falta. Si no los tenés todavía, el notebook sigue funcionando con datos sintéticos "
          "para que puedas validar que el pipeline corre de punta a punta.")

print(f"\nRuta features tabulares: {RUTA_FEATURES_TABULARES}  "
      f"(existe: {os.path.exists(RUTA_FEATURES_TABULARES)})")
print(f"Carpeta TDI por cohorte: {CARPETA_TDI_COHORTE}  "
      f"(subcarpetas encontradas: {len(glob.glob(CARPETA_TDI_COHORTE + '*/'))})")


## **0.2. Ingesta de la cohorte por paciente y control (≈70 vs. ≈30)**

**Flujo de trabajo (el Archivo 2 se usa sin cambios):**

1. **Manifiesto.** La celda 0.2.a crea la plantilla `manifiesto_cohorte.csv` con 70 filas `EPI001…EPI070` y 30 filas `CTL001…CTL030` (si ya existe, no la sobrescribe). Complétala con `edad`, `sexo`, `lateralidad_foco` (izquierda / derecha / bilateral; vacío en controles) y `sitio_adquisicion`. Puedes renombrar los `subject_id`, pero deben coincidir con los nombres de los zips.
2. **Ejecutar el Archivo 2 por sujeto**, apuntando a los `.tt.gz` y `.fz` de ese sujeto.
3. **Empaquetar la salida de ese sujeto** como `<subject_id>.zip`. Se puede pegar esta celda al final de la sesión del Archivo 2, sin tocar el notebook:
   ```python
   import zipfile, glob, os
   SUJETO = "EPI001"                       # el mismo subject_id del manifiesto
   NECESARIOS = ["geometria_clusters.csv", "mapeo_cluster_tracto.csv",
                 "contraste_interhemisferico.csv", "indices_lateralidad.csv"]
   with zipfile.ZipFile(f"/content/{SUJETO}.zip", "w", zipfile.ZIP_DEFLATED) as z:
       for f in NECESARIOS:
           z.write(f"/content/{f}", f)
       for f in sorted(glob.glob("/content/tdi_grises/*_gris.png")):
           z.write(f, "tdi_grises/" + os.path.basename(f))
   ```
4. **Subir los zips** a una carpeta (`MODO_CARGA = "carpeta"`, por defecto `cohorte_zips/`), a Google Drive (`"drive"`) o con el selector de archivos de Colab (`"subir"`).
5. **Ejecutar 0.2.c**, que extrae cada zip, valida la cohorte y arma la tabla de métricas geométricas por sujeto.

Si un sujeto trae solo `tdi_grises/` sin los CSV, entra a la sección A (CNN) pero no a la B (métricas); la tabla de validación lo indica.

In [ ]:
# 0.2.a — Parámetros de la cohorte y plantilla de manifiesto
import hashlib
import shutil
import zipfile

GRUPO_EPI, GRUPO_CTRL = "epilepsia", "control"
N_EPI_ESPERADO, N_CTRL_ESPERADO = 70, 30   # tamaño muestral objetivo (≈70 pacientes vs. ≈30 controles)
TOLERANCIA_N = 0.10                         # aviso si el n real se desvía más de 10 % del objetivo
MIN_POR_CLASE = 8                           # por debajo de esto no se puede validar de forma razonable
EXIGIR_8_TRACTOS = True                     # un sujeto sin los 8 clústeres (4 tractos x 2 lados) se excluye

MODO_CARGA = "carpeta"                      # "carpeta" | "drive" | "subir" (selector de archivos de Colab)
FUENTE_COHORTE = CARPETA_SALIDA + "cohorte_zips/"      # un <subject_id>.zip (o carpeta) por sujeto
RUTA_DRIVE_COHORTE = "/content/drive/MyDrive/cohorte_zips/"
RUTA_MANIFIESTO = CARPETA_SALIDA + "manifiesto_cohorte.csv"

TRACTOS_CNN = ["arc", "unc", "mdlf", "ilf"]   # tractos que entran a la CNN (Sección A)
LADOS_CNN = ["l", "r"]

COLUMNAS_MANIFIESTO = ["subject_id", "grupo_dx", "edad", "sexo", "lateralidad_foco", "sitio_adquisicion"]


def crear_plantilla_manifiesto(ruta, n_epi=N_EPI_ESPERADO, n_ctrl=N_CTRL_ESPERADO):
    """Crea la plantilla (sin sobrescribir una existente). Solo el identificador y el grupo
    vienen rellenados; edad, sexo, lateralidad y sitio los completa el usuario."""
    if os.path.exists(ruta):
        print(f"El manifiesto ya existe y no se sobrescribe: {ruta}")
        return
    filas = ([{"subject_id": f"EPI{i:03d}", "grupo_dx": GRUPO_EPI} for i in range(1, n_epi + 1)] +
             [{"subject_id": f"CTL{i:03d}", "grupo_dx": GRUPO_CTRL} for i in range(1, n_ctrl + 1)])
    pd.DataFrame(filas, columns=COLUMNAS_MANIFIESTO).to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Plantilla creada: {ruta}  ({n_epi} epilepsia + {n_ctrl} control). Complete las covariables.")


crear_plantilla_manifiesto(RUTA_MANIFIESTO)
os.makedirs(FUENTE_COHORTE, exist_ok=True)

if IN_COLAB and MODO_CARGA == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    FUENTE_COHORTE = RUTA_DRIVE_COHORTE
elif IN_COLAB and MODO_CARGA == "subir":
    from google.colab import files
    print("Seleccione los .zip de los sujetos (nombre = subject_id.zip) y, si hace falta, el manifiesto .csv:")
    for nombre, contenido in files.upload().items():
        destino_subida = RUTA_MANIFIESTO if nombre.lower().endswith(".csv") else os.path.join(FUENTE_COHORTE, nombre)
        with open(destino_subida, "wb") as f:
            f.write(contenido)

print(f"Fuente de la cohorte: {FUENTE_COHORTE}  (elementos: {len(glob.glob(FUENTE_COHORTE + '*'))})")

In [ ]:
# 0.2.b — Funciones de ingesta, validación y extracción de métricas geométricas
from scipy import stats

SINONIMOS_GRUPO = {
    "epilepsia": GRUPO_EPI, "epi": GRUPO_EPI, "paciente": GRUPO_EPI, "tle": GRUPO_EPI, "elt": GRUPO_EPI,
    "epilepsy": GRUPO_EPI, "patient": GRUPO_EPI,
    "control": GRUPO_CTRL, "ctl": GRUPO_CTRL, "hc": GRUPO_CTRL, "sano": GRUPO_CTRL, "healthy": GRUPO_CTRL,
}
ARCHIVOS_REQUERIDOS = ["geometria_clusters.csv", "mapeo_cluster_tracto.csv"]
ARCHIVOS_OPCIONALES = ["contraste_interhemisferico.csv", "indices_lateralidad.csv"]
CODIGOS_TRACTO = {"Arcuato": "arc", "Uncinado": "unc", "MdLF": "mdlf", "ILF": "ilf"}
MAPA_METRICA_LI = {"λ₁": "l1", "λ₂": "l2", "λ₃": "l3", "λ₁/λ₂": "elong", "λ₁/λ₃": "esf",
                   "longitud media": "long", "n streamlines (sin IC)": "n"}


def normalizar_id(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def leer_manifiesto(ruta):
    """Lee y valida el manifiesto. Detiene la ejecución ante etiquetas no válidas o ids repetidos."""
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe el manifiesto: {ruta}")
    m = pd.read_csv(ruta, dtype=str).apply(lambda c: c.str.strip())
    faltan = [c for c in ("subject_id", "grupo_dx") if c not in m.columns]
    if faltan:
        raise ValueError(f"Al manifiesto le faltan las columnas obligatorias: {faltan}")
    m = m.dropna(subset=["subject_id"]).reset_index(drop=True)
    m["grupo_dx"] = m["grupo_dx"].str.lower().map(SINONIMOS_GRUPO)
    if m["grupo_dx"].isna().any():
        raise ValueError("grupo_dx no válido (use 'epilepsia' o 'control') en: "
                         f"{m.loc[m['grupo_dx'].isna(), 'subject_id'].tolist()}")
    ids = m["subject_id"].map(normalizar_id)
    if ids.duplicated().any():
        raise ValueError(f"subject_id repetido en el manifiesto: {m.loc[ids.duplicated(keep=False), 'subject_id'].tolist()}")
    for c in COLUMNAS_MANIFIESTO:
        if c not in m.columns:
            m[c] = np.nan
    return m


def extraer_zip_seguro(ruta_zip, destino):
    """Extrae el zip en 'destino' (reemplazando lo anterior) y rechaza rutas que salgan de ahí."""
    if os.path.isdir(destino):
        shutil.rmtree(destino)
    os.makedirs(destino, exist_ok=True)
    base = os.path.abspath(destino)
    with zipfile.ZipFile(ruta_zip) as z:
        for miembro in z.namelist():
            objetivo = os.path.abspath(os.path.join(destino, miembro))
            if objetivo != base and not objetivo.startswith(base + os.sep):
                raise ValueError(f"Ruta insegura dentro de {ruta_zip}: {miembro}")
        z.extractall(destino)


def localizar_fuente_sujeto(fuente_dir, subject_id):
    objetivo = normalizar_id(subject_id)
    for ruta in sorted(glob.glob(os.path.join(fuente_dir, "*"))):
        base = os.path.basename(ruta)
        es_zip = base.lower().endswith(".zip")
        if (es_zip or os.path.isdir(ruta)) and normalizar_id(base[:-4] if es_zip else base) == objetivo:
            return ruta
    return None


def _por_profundidad(rutas):
    return sorted(rutas, key=lambda p: (p.count(os.sep), p))


def buscar_archivo(carpeta, nombre):
    r = _por_profundidad(glob.glob(os.path.join(carpeta, "**", nombre), recursive=True))
    return r[0] if r else None


def buscar_dir_tdi(carpeta):
    r = _por_profundidad([p for p in glob.glob(os.path.join(carpeta, "**", "tdi_grises"), recursive=True)
                          if os.path.isdir(p)])
    return r[0] if r else None


def ingestar_cohorte(manifiesto, fuente_dir, destino, corte="axial"):
    """Para cada sujeto del manifiesto localiza su zip/carpeta, lo extrae en destino/<subject_id>/
    y comprueba qué salidas del Archivo 2 trae. Devuelve una fila de estado por sujeto."""
    estados = []
    usados = set()
    for fila in manifiesto.itertuples():
        sid = fila.subject_id
        carpeta = os.path.join(destino, sid)
        problema = []
        fuente = localizar_fuente_sujeto(fuente_dir, sid)
        if fuente is None:
            problema.append("sin zip/carpeta en la fuente")
        else:
            usados.add(os.path.abspath(fuente))
            if fuente.lower().endswith(".zip"):
                extraer_zip_seguro(fuente, carpeta)
            elif os.path.abspath(fuente) != os.path.abspath(carpeta):
                if os.path.isdir(carpeta):
                    shutil.rmtree(carpeta)
                shutil.copytree(fuente, carpeta)

        carpeta_ok = os.path.isdir(carpeta)
        dir_tdi = buscar_dir_tdi(carpeta) if carpeta_ok else None
        n_img = len(glob.glob(os.path.join(dir_tdi, f"*_{corte}_gris.png"))) if dir_tdi else 0
        rutas = {n: (buscar_archivo(carpeta, n) if carpeta_ok else None)
                 for n in ARCHIVOS_REQUERIDOS + ARCHIVOS_OPCIONALES}
        for n in ARCHIVOS_REQUERIDOS:
            if rutas[n] is None:
                problema.append(f"falta {n}")
        if n_img == 0:
            problema.append(f"sin imágenes *_{corte}_gris.png")
        huella = None
        if rutas["geometria_clusters.csv"]:
            with open(rutas["geometria_clusters.csv"], "rb") as f:
                huella = hashlib.md5(f.read()).hexdigest()
        estados.append({"subject_id": sid, "grupo_dx": fila.grupo_dx, "n_imagenes": n_img,
                        "tiene_geometria": rutas["geometria_clusters.csv"] is not None,
                        "tiene_mapeo": rutas["mapeo_cluster_tracto.csv"] is not None,
                        "tiene_contraste": rutas["contraste_interhemisferico.csv"] is not None,
                        "tiene_lateralidad": rutas["indices_lateralidad.csv"] is not None,
                        "huella_geometria": huella, "problema": "; ".join(problema)})

    df = pd.DataFrame(estados)
    # la misma salida del Archivo 2 subida bajo dos identificadores es un error de carga: se excluyen ambos
    dup = df["huella_geometria"].notna() & df["huella_geometria"].duplicated(keep=False)
    for h in df.loc[dup, "huella_geometria"].unique():
        ids = df.loc[df["huella_geometria"] == h, "subject_id"].tolist()
        df.loc[df["huella_geometria"] == h, "problema"] += f"; salida idéntica en {ids}"
    df["problema"] = df["problema"].str.strip("; ")
    huerfanos = [os.path.basename(p) for p in sorted(glob.glob(os.path.join(fuente_dir, "*")))
                 if os.path.abspath(p) not in usados and (p.lower().endswith(".zip") or os.path.isdir(p))]
    return df, huerfanos


def extraer_features_geometricas(carpeta_sujeto):
    """Una fila ancha (dict) con las métricas del Archivo 2: geometría por tracto y lado,
    tortuosidad mediana y índices de lateralidad. Lee solo CSV exportados por el Archivo 2."""
    geo = pd.read_csv(buscar_archivo(carpeta_sujeto, "geometria_clusters.csv"))
    mapeo = pd.read_csv(buscar_archivo(carpeta_sujeto, "mapeo_cluster_tracto.csv"))
    m = geo.merge(mapeo[["cluster_id", "variable"]], on="cluster_id", validate="one_to_one")
    f = {}
    for r in m.itertuples():
        p = f"geo_{r.variable}"
        f[f"{p}_n_streamlines"] = r.n_streamlines
        f[f"{p}_long_mm"] = r.longitud_media_mm
        f[f"{p}_l1"], f[f"{p}_l2"], f[f"{p}_l3"] = r.eigenvalor_1, r.eigenvalor_2, r.eigenvalor_3
        f[f"{p}_elong"] = r.elongacion
        f[f"{p}_esf"] = r.relacion_aspecto_esfericidad
        f[f"{p}_ori_ap"] = abs(r.orientacion_y)
    ruta_c = buscar_archivo(carpeta_sujeto, "contraste_interhemisferico.csv")
    if ruta_c:
        c = pd.read_csv(ruta_c)
        for r in c[c["métrica"] == "tortuosidad"].itertuples():
            cod = CODIGOS_TRACTO.get(r.tracto)
            if cod:
                f[f"geo_tracto_{cod}_l_tort"], f[f"geo_tracto_{cod}_r_tort"] = r.mediana_izq, r.mediana_der
    ruta_l = buscar_archivo(carpeta_sujeto, "indices_lateralidad.csv")
    if ruta_l:
        li = pd.read_csv(ruta_l)
        for r in li.itertuples():
            cod, met = CODIGOS_TRACTO.get(r.tracto), MAPA_METRICA_LI.get(r.métrica)
            if cod and met:
                f[f"geo_li_{cod}_{met}"] = r.LI
    return f


def comparar_covariables(df):
    """Compara edad y sexo entre grupos: un desbalance es una fuente de confusión del clasificador."""
    filas = []
    if "edad" in df:
        e = pd.to_numeric(df["edad"], errors="coerce")
        a, b = e[df["grupo_dx"] == GRUPO_EPI].dropna(), e[df["grupo_dx"] == GRUPO_CTRL].dropna()
        if len(a) > 2 and len(b) > 2:
            filas.append({"covariable": "edad (mediana epilepsia vs control)",
                          "valores": f"{a.median():.1f} vs {b.median():.1f}",
                          "p (Mann–Whitney)": stats.mannwhitneyu(a, b).pvalue})
    if "sexo" in df and df["sexo"].notna().sum() > 4:
        tabla = pd.crosstab(df["grupo_dx"], df["sexo"])
        if tabla.shape[0] == 2 and tabla.shape[1] >= 2:
            p = (stats.fisher_exact(tabla.values)[1] if tabla.shape == (2, 2)
                 else stats.chi2_contingency(tabla.values)[1])
            filas.append({"covariable": "sexo", "valores": tabla.to_dict("index").__repr__(),
                          "p (Mann–Whitney)": p})
    return pd.DataFrame(filas)

In [ ]:
# 0.2.c — Ejecutar la ingesta, validar la cohorte y construir las métricas geométricas por sujeto
RUTA_MANIFIESTO_OK = os.path.exists(RUTA_MANIFIESTO)
HAY_FUENTE = len(glob.glob(FUENTE_COHORTE + "*")) > 0

if RUTA_MANIFIESTO_OK and HAY_FUENTE:
    manifiesto = leer_manifiesto(RUTA_MANIFIESTO)
    df_estado, huerfanos = ingestar_cohorte(manifiesto, FUENTE_COHORTE, CARPETA_TDI_COHORTE)

    # ---- criterio de inclusión
    df_estado["ok"] = (df_estado["problema"] == "")
    if EXIGIR_8_TRACTOS:
        for i, fila in df_estado.iterrows():
            if fila["ok"] and fila["tiene_mapeo"]:
                n_cl = len(pd.read_csv(buscar_archivo(os.path.join(CARPETA_TDI_COHORTE, fila["subject_id"]),
                                                      "mapeo_cluster_tracto.csv")))
                if n_cl != 8:
                    df_estado.loc[i, ["ok", "problema"]] = [False, f"{n_cl} clústeres mapeados (se esperaban 8)"]

    print("=" * 80)
    print("INGESTA DE LA COHORTE")
    print("=" * 80)
    tabla_estado = df_estado.drop(columns="huella_geometria")
    if len(tabla_estado) <= 20:
        print(tabla_estado.to_string(index=False))
    else:
        print(f"{len(tabla_estado)} sujetos en el manifiesto; detalle completo en cohorte_validacion.csv")
        print(tabla_estado.groupby("grupo_dx").agg(
            sujetos=("subject_id", "size"), completos=("ok", "sum"),
            imagenes_por_sujeto_min=("n_imagenes", "min"), imagenes_por_sujeto_max=("n_imagenes", "max")).to_string())
    if huerfanos:
        print(f"\n⚠ Zips o carpetas en la fuente que NO están en el manifiesto: {huerfanos}")

    # ---- conteos frente al objetivo y línea base
    n_epi = int(((df_estado["grupo_dx"] == GRUPO_EPI) & df_estado["ok"]).sum())
    n_ctl = int(((df_estado["grupo_dx"] == GRUPO_CTRL) & df_estado["ok"]).sum())
    print(f"\nSujetos utilizables: epilepsia = {n_epi} (objetivo {N_EPI_ESPERADO}), "
          f"control = {n_ctl} (objetivo {N_CTRL_ESPERADO})")
    for nombre, n, objetivo in ((GRUPO_EPI, n_epi, N_EPI_ESPERADO), (GRUPO_CTRL, n_ctl, N_CTRL_ESPERADO)):
        if abs(n - objetivo) > TOLERANCIA_N * objetivo:
            print(f"⚠ {nombre}: {n} utilizables, más de {TOLERANCIA_N:.0%} de diferencia con el objetivo ({objetivo}).")
    if min(n_epi, n_ctl) < MIN_POR_CLASE:
        raise ValueError(f"Hay menos de {MIN_POR_CLASE} sujetos utilizables en alguna clase "
                         f"(epilepsia={n_epi}, control={n_ctl}). Corrija la ingesta antes de modelar.")
    base = max(n_epi, n_ctl) / (n_epi + n_ctl)
    print(f"Línea base: un clasificador que siempre predice la clase mayoritaria acierta {base:.0%}; "
          f"por eso la exactitud no es la métrica principal (AUC, exactitud balanceada, MCC).")

    excluidos = df_estado.loc[~df_estado["ok"], ["subject_id", "grupo_dx", "problema"]]
    if len(excluidos):
        print(f"\nExcluidos ({len(excluidos)}):")
        print(excluidos.to_string(index=False))

    df_cohorte_ok = (manifiesto.merge(df_estado.loc[df_estado["ok"], ["subject_id", "n_imagenes",
                                                                       "tiene_contraste", "tiene_lateralidad"]],
                                      on="subject_id", how="inner").reset_index(drop=True))

    cov = comparar_covariables(df_cohorte_ok)
    if len(cov):
        print("\nBalance de covariables entre grupos:")
        print(cov.round(4).to_string(index=False))
        if (cov["p (Mann–Whitney)"] < 0.05).any():
            print("⚠ Hay diferencias entre grupos en edad o sexo: son posibles variables de confusión del "
                  "clasificador; considere emparejar o ajustar.")

    # ---- métricas geométricas por sujeto (formato ancho)
    filas_geo = []
    for fila in df_cohorte_ok.itertuples():
        try:
            f = extraer_features_geometricas(os.path.join(CARPETA_TDI_COHORTE, fila.subject_id))
            f["subject_id"] = fila.subject_id
            filas_geo.append(f)
        except Exception as e:
            print(f"⚠ {fila.subject_id}: no se pudieron extraer las métricas geométricas ({e!r})")
    df_geom_wide = pd.DataFrame(filas_geo)
    df_estado.drop(columns="huella_geometria").to_csv(CARPETA_SALIDA + "cohorte_validacion.csv", index=False,
                                                       encoding="utf-8-sig")
    df_geom_wide.to_csv(CARPETA_SALIDA + "features_geometricas_cohorte.csv", index=False, encoding="utf-8-sig")
    print(f"\nMétricas geométricas: {df_geom_wide.shape[0]} sujetos x {df_geom_wide.shape[1] - 1} variables "
          f"(guardadas en features_geometricas_cohorte.csv)")
else:
    print("No hay cohorte real cargada (falta el manifiesto o la carpeta de zips está vacía). "
          "Las Secciones A y B usarán datos SINTÉTICOS solo para validar que el pipeline corre; "
          "esos resultados no tienen valor científico.")

# **Sección A — CNN sobre Imágenes TDI**

## **A.1. Formulación del enfoque visual**
Las métricas escalares (FA, MD, volumen) resumen cada tracto en un puñado de números, perdiendo su
forma espacial completa. Para aprovechar esa información geométrica, se entrena una **Red Neuronal
Convolucional (CNN)** directamente sobre los cortes TDI en escala de grises que exporta el Archivo 2
(Sección 6.2) — una imagen 2D por clúster y por corte anatómico, que conserva la morfología real del
tracto (curvatura, elongación, grosor) en vez de reducirla a un promedio.

## **A.2. Unidad de análisis: corte por clúster, no por sujeto**
Cada sujeto aporta varias imágenes (hasta 4 tractos × 2 lados = 8 clústeres, un corte axial cada
uno). Se entrena la CNN a **nivel de clúster** (cada imagen es una muestra independiente, etiquetada
con el `grupo_dx` de su sujeto), lo cual es una práctica habitual en imagenología médica con N de
pacientes reducido (más muestras de entrenamiento que sujetos). La partición train/validación/prueba se hace **estratificada por diagnóstico y agrupada por `subject_id`** (`StratifiedGroupKFold`) — para que ninguna
imagen de un sujeto del conjunto de prueba haya sido vista, aunque sea de otro tracto, durante el
entrenamiento.


In [ ]:
IMG_RE = re.compile(r"^tracto_(?P<cod>[a-z]+)_(?P<lado>[lr])_cluster(?P<cid>\d+)_(?P<corte>[a-z]+)_gris\.png$")


def listar_imagenes_tdi_cohorte(df_cohorte, carpeta_cohorte, corte="axial", tractos=None, lados=None):
    """Registros {ruta, subject_id, grupo_dx, tracto, lado, cluster_id} para cada imagen TDI de los
    sujetos validados. grupo_dx viene SIEMPRE del manifiesto."""
    registros, omitidas = [], []
    for fila in df_cohorte.itertuples():
        dir_tdi = buscar_dir_tdi(os.path.join(carpeta_cohorte, fila.subject_id))
        for ruta in sorted(glob.glob(os.path.join(dir_tdi, f"*_{corte}_gris.png"))) if dir_tdi else []:
            m = IMG_RE.match(os.path.basename(ruta))
            if not m or m.group("corte") != corte:
                omitidas.append(os.path.basename(ruta))
                continue
            if (tractos and m.group("cod") not in tractos) or (lados and m.group("lado") not in lados):
                continue
            registros.append({"ruta": ruta, "subject_id": fila.subject_id, "grupo_dx": fila.grupo_dx,
                              "tracto": m.group("cod"), "lado": m.group("lado"),
                              "cluster_id": int(m.group("cid"))})
    if omitidas:
        print(f"⚠ {len(omitidas)} imágenes con nombre fuera de la nomenclatura tracto_<código>_<lado>_cluster<NN> "
              f"fueron omitidas (p. ej. {omitidas[:3]}).")
    return pd.DataFrame(registros)


if "df_cohorte_ok" in globals():
    df_imagenes = listar_imagenes_tdi_cohorte(df_cohorte_ok, CARPETA_TDI_COHORTE, corte="axial",
                                              tractos=TRACTOS_CNN, lados=LADOS_CNN)
else:
    df_imagenes = pd.DataFrame()

print(f"Imágenes TDI (corte axial) encontradas: {len(df_imagenes)}")
if len(df_imagenes):
    print(f"Sujetos: {df_imagenes['subject_id'].nunique()}  "
          f"(imágenes por grupo: {df_imagenes['grupo_dx'].value_counts().to_dict()})")
df_imagenes.head()

## **A.4. Carga de imágenes (o generación de un set sintético de validación)**
Si no se encontraron imágenes reales (Sección A.3 vacía), se genera un set sintético **con señal
controlada y claramente marcado como tal** — nunca etiquetas aleatorias desconectadas de los datos —
únicamente para verificar que el resto del pipeline (carga → CNN → evaluación) corre sin errores.

In [ ]:
from PIL import Image

TAMANO_IMG = 128  # input_shape de la CNN (Sección A.5): (TAMANO_IMG, TAMANO_IMG, 1)


def cargar_y_redimensionar(ruta, tamano=TAMANO_IMG):
    """Carga una imagen en escala de grises, la redimensiona a (tamano, tamano)
    y normaliza su intensidad a [0, 1].
    """
    img = Image.open(ruta).convert("L").resize((tamano, tamano))
    return np.asarray(img, dtype=np.float32) / 255.0


def generar_set_sintetico_imagenes(n_sujetos=40, n_clusters_por_sujeto=4, tamano=TAMANO_IMG,
                                    semilla=42):
    """Genera imágenes sintéticas con una señal controlada y conocida (un
    gradiente diagonal cuya pendiente difiere, en promedio, entre 'epilepsia'
    y 'control', más ruido gaussiano) -- NUNCA etiquetas desconectadas de los
    datos. Sirve solo para validar que el pipeline de carga/entrenamiento/
    evaluación corre de punta a punta antes de tener imágenes reales.
    """
    rng = np.random.default_rng(semilla)
    xs, ys, subject_ids, grupos = [], [], [], []
    coord = np.linspace(0, 1, tamano)
    malla_x, malla_y = np.meshgrid(coord, coord)

    for i in range(n_sujetos):
        grupo = "epilepsia" if i % 2 == 0 else "control"
        pendiente = 1.6 if grupo == "epilepsia" else 1.0
        for c in range(n_clusters_por_sujeto):
            patron = np.sin(pendiente * (malla_x + malla_y) * np.pi) * 0.4 + 0.5
            ruido = rng.normal(0, 0.05, size=(tamano, tamano))
            xs.append(np.clip(patron + ruido, 0, 1).astype(np.float32))
            subject_ids.append(f"SIM{i:03d}")
            grupos.append(grupo)

    return np.array(xs), np.array(subject_ids), np.array(grupos)


if len(df_imagenes):
    X_img = np.stack([cargar_y_redimensionar(r) for r in df_imagenes["ruta"]])
    subject_ids_img = df_imagenes["subject_id"].values
    grupos_img = df_imagenes["grupo_dx"].values
    print(f"Imágenes reales cargadas: {X_img.shape}")
else:
    if "df_cohorte_ok" in globals():
        raise RuntimeError("Hay una cohorte validada pero ninguna imagen TDI que cumpla la nomenclatura; "
                           "revise la Sección 0.2 y los nombres de archivo.")
    print("No se encontraron imágenes reales en CARPETA_TDI_COHORTE; generando set SINTÉTICO "
          "de validación (ver nota metodológica más arriba)...")
    X_img, subject_ids_img, grupos_img = generar_set_sintetico_imagenes()
    print(f"Imágenes sintéticas generadas: {X_img.shape}")

X_img = X_img[..., np.newaxis]  # (N, TAMANO_IMG, TAMANO_IMG, 1) -- canal único (escala de grises)
assert set(np.unique(grupos_img)) <= {"epilepsia", "control"}, \
    f"Etiquetas no válidas en grupos_img: {set(np.unique(grupos_img))}"
y_img = (grupos_img == "epilepsia").astype(int)
print(f"Forma final de X_img: {X_img.shape}   Distribución de clases: "
      f"control={np.sum(y_img == 0)}, epilepsia={np.sum(y_img == 1)}")


## **A.5. Arquitectura de la CNN**
Tres bloques convolucionales (filtros 3×3, `BatchNormalization`, `MaxPooling2D`) seguidos de una
capa densa con `Dropout` (40%) y una salida sigmoide para clasificación binaria.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


def construir_modelo_cnn(input_shape=(TAMANO_IMG, TAMANO_IMG, 1)):
    """Construye una CNN para clasificar cortes TDI (epilepsia vs. control)."""
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


modelo_cnn = construir_modelo_cnn()
modelo_cnn.summary()


## **A.6. Partición estratificada y agrupada por sujeto, entrenamiento y evaluación**

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              ConfusionMatrixDisplay)


def particion_estratificada_por_sujeto(y, grupos, n_splits, semilla=42):
    """Primer pliegue de un StratifiedGroupKFold: conserva la proporción de clases y mantiene todas las
    imágenes de un sujeto del mismo lado de la partición."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=semilla)
    return next(sgkf.split(np.zeros(len(y)), y, groups=grupos))


# prueba: ≈25 % de los sujetos; validación (parada temprana): ≈20 % de los sujetos del entrenamiento
idx_train_img, idx_test_img = particion_estratificada_por_sujeto(y_img, subject_ids_img, n_splits=4)
rel_fit, rel_val = particion_estratificada_por_sujeto(y_img[idx_train_img], subject_ids_img[idx_train_img], n_splits=5)
idx_fit, idx_val = idx_train_img[rel_fit], idx_train_img[rel_val]

X_train_img, X_test_img = X_img[idx_train_img], X_img[idx_test_img]
y_train_img, y_test_img = y_img[idx_train_img], y_img[idx_test_img]
X_fit, y_fit = X_img[idx_fit], y_img[idx_fit]
X_val, y_val = X_img[idx_val], y_img[idx_val]


def resumen_particion(nombre, idx):
    sid = pd.Series(subject_ids_img[idx]).drop_duplicates()
    cls = pd.Series(y_img[idx], index=subject_ids_img[idx])
    por_clase = cls.groupby(level=0).first().value_counts().to_dict()
    print(f"  {nombre:<14} {len(idx):>5} imágenes | {len(sid):>3} sujetos "
          f"(control={por_clase.get(0, 0)}, epilepsia={por_clase.get(1, 0)})")


print("Partición estratificada y agrupada por sujeto:")
resumen_particion("entrenamiento", idx_fit)
resumen_particion("validación", idx_val)
resumen_particion("prueba", idx_test_img)
assert not (set(subject_ids_img[idx_fit]) & set(subject_ids_img[idx_test_img])), "sujetos repetidos entre train y test"
assert not (set(subject_ids_img[idx_val]) & set(subject_ids_img[idx_fit])), "sujetos repetidos entre train y validación"
assert len(set(y_img[idx_test_img])) == 2 and len(set(y_img[idx_val])) == 2, "alguna partición tiene una sola clase"

# Peso balanceado por clase (con 70:30 la clase minoritaria debe pesar más)
n_pos, n_neg = y_fit.sum(), len(y_fit) - y_fit.sum()
peso_clases = {0: 1.0, 1: (n_neg / max(n_pos, 1))} if n_pos > 0 else {0: 1.0, 1: 1.0}

callback_parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)

historial = modelo_cnn.fit(
    X_fit, y_fit,
    epochs=60, batch_size=8, validation_data=(X_val, y_val),
    class_weight=peso_clases, callbacks=[callback_parada_temprana], verbose=0,
)
print(f"Entrenamiento finalizado en {len(historial.history['loss'])} épocas "
      f"(parada temprana monitoreando val_loss en sujetos de validación no usados para ajustar).")

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(historial.history["loss"], label="Entrenamiento")
plt.plot(historial.history["val_loss"], label="Validación")
plt.title("Pérdida (binary cross-entropy)")
plt.xlabel("Época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historial.history["accuracy"], label="Entrenamiento")
plt.plot(historial.history["val_accuracy"], label="Validación")
plt.title("Accuracy")
plt.xlabel("Época")
plt.legend()

plt.tight_layout()
ruta_curvas_png = CARPETA_SALIDA + "cnn_curvas_entrenamiento.png"
plt.savefig(ruta_curvas_png, dpi=140, bbox_inches="tight")
plt.show()
print(f"Curvas guardadas en: {ruta_curvas_png}")


In [ ]:
y_proba_cnn = modelo_cnn.predict(X_test_img, verbose=0).ravel()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)

print("=== CNN — Evaluación a nivel de imagen (conjunto de prueba, sujetos no vistos) ===")
print(classification_report(y_test_img, y_pred_cnn,
                             target_names=["Control (0)", "Epilepsia (1)"], zero_division=0))

if len(set(y_test_img)) > 1:
    auc_cnn = roc_auc_score(y_test_img, y_proba_cnn)
    print(f"AUC-ROC: {auc_cnn:.3f}")
else:
    auc_cnn = float("nan")
    print("AUC-ROC no definido: el conjunto de prueba solo tiene una clase.")

matriz_confusion_cnn = confusion_matrix(y_test_img, y_pred_cnn)
ConfusionMatrixDisplay(matriz_confusion_cnn, display_labels=["Control", "Epilepsia"]).plot(
    cmap="Blues", colorbar=False
)
plt.title("CNN — Matriz de confusión (nivel de imagen)")
ruta_matriz_cnn_png = CARPETA_SALIDA + "cnn_matriz_confusion.png"
plt.savefig(ruta_matriz_cnn_png, dpi=140, bbox_inches="tight")
plt.show()


## **A.7. Agregación a nivel de sujeto (voto mayoritario)**
La Sección A.6 evalúa a nivel de imagen (clúster). Para obtener una decisión clínica por **paciente**
—la unidad que realmente importa—, se agregan las predicciones de todas las imágenes de prueba de un
mismo sujeto por voto mayoritario de probabilidad promedio.

In [ ]:
df_pred_img = pd.DataFrame({
    "subject_id": subject_ids_img[idx_test_img],
    "y_real": y_test_img,
    "y_proba": y_proba_cnn,
})

df_pred_sujeto = df_pred_img.groupby("subject_id").agg(
    y_real=("y_real", "first"), y_proba_media=("y_proba", "mean")
).reset_index()
df_pred_sujeto["y_pred"] = (df_pred_sujeto["y_proba_media"] >= 0.5).astype(int)

print("=== CNN — Evaluación agregada a nivel de SUJETO (voto por promedio de probabilidad) ===")
print(classification_report(df_pred_sujeto["y_real"], df_pred_sujeto["y_pred"],
                             target_names=["Control (0)", "Epilepsia (1)"], zero_division=0))
df_pred_sujeto.round(3)


## **A.7b. Intervalos de confianza por bootstrap de sujetos**
Con una sola partición de prueba (≈25 sujetos), las métricas puntuales son muy variables. Se remuestrean **sujetos** (no imágenes) con reemplazo para estimar un intervalo de confianza al 95 % de cada métrica. Con clases 70:30 se reportan además la exactitud balanceada y el MCC, que no se inflan por la clase mayoritaria.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef


def ic_bootstrap_sujetos(y_real, y_score, umbral=0.5, B=2000, semilla=42):
    """Métrica puntual e IC95 % (percentil) remuestreando sujetos con reemplazo."""
    rng = np.random.default_rng(semilla)
    y_real, y_score = np.asarray(y_real), np.asarray(y_score)
    n = len(y_real)

    def metricas(idx):
        yr, ys = y_real[idx], y_score[idx]
        if len(set(yr)) < 2:
            return None
        yp = (ys >= umbral).astype(int)
        tn, fp, fn, tp = confusion_matrix(yr, yp, labels=[0, 1]).ravel()
        return {"AUC": roc_auc_score(yr, ys), "Exactitud balanceada": balanced_accuracy_score(yr, yp),
                "MCC": matthews_corrcoef(yr, yp),
                "Sensibilidad": tp / (tp + fn) if (tp + fn) else np.nan,
                "Especificidad": tn / (tn + fp) if (tn + fp) else np.nan}

    puntual = metricas(np.arange(n))
    reps = [m for m in (metricas(rng.integers(0, n, n)) for _ in range(B)) if m]
    filas = []
    for k, v in puntual.items():
        vals = np.array([r[k] for r in reps], dtype=float)
        filas.append({"métrica": k, "estimado": v, "IC95_inf": np.nanpercentile(vals, 2.5),
                      "IC95_sup": np.nanpercentile(vals, 97.5)})
    return pd.DataFrame(filas), len(reps)


if "df_pred_sujeto" in globals():
    df_ic_cnn, n_validas = ic_bootstrap_sujetos(df_pred_sujeto["y_real"], df_pred_sujeto["y_proba_media"])
    print(f"CNN — nivel de sujeto (n = {len(df_pred_sujeto)}; {n_validas} remuestreos válidos)")
    display(df_ic_cnn.round(3))
    df_ic_cnn.to_csv(CARPETA_SALIDA + "cnn_metricas_ic95_sujetos.csv", index=False, encoding="utf-8-sig")

## **A.7c. Validación cruzada agrupada de la CNN (opcional)**
Es la evaluación más confiable de la CNN con esta muestra, porque cada sujeto actúa una vez como prueba. Es más lenta (un entrenamiento por pliegue), por eso viene desactivada: cambie `EJECUTAR_CV_CNN = True` para correrla.

In [ ]:
EJECUTAR_CV_CNN = False   # True: validación cruzada estratificada y agrupada por sujeto (5 pliegues)


def evaluar_cnn_cv(construir_modelo, X, y, grupos, n_splits=5, semilla=42, epochs=60, batch_size=8,
                   callbacks_factory=lambda: [], limpiar=lambda: None):
    """Devuelve el AUC, la exactitud balanceada y el MCC por pliegue, a nivel de sujeto."""
    filas = []
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=semilla)
    for k, (tr, te) in enumerate(sgkf.split(np.zeros(len(y)), y, groups=grupos), 1):
        rel_fit, rel_val = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=k)
                                .split(np.zeros(len(tr)), y[tr], groups=grupos[tr]))
        fit, val = tr[rel_fit], tr[rel_val]
        npos = y[fit].sum()
        pesos = {0: 1.0, 1: (len(fit) - npos) / max(npos, 1)}
        modelo = construir_modelo()
        modelo.fit(X[fit], y[fit], epochs=epochs, batch_size=batch_size, validation_data=(X[val], y[val]),
                   class_weight=pesos, callbacks=callbacks_factory(), verbose=0)
        d = pd.DataFrame({"sid": grupos[te], "y": y[te], "p": modelo.predict(X[te], verbose=0).ravel()})
        s = d.groupby("sid").agg(y=("y", "first"), p=("p", "mean"))
        yp = (s["p"] >= 0.5).astype(int)
        filas.append({"pliegue": k, "n_sujetos_test": len(s),
                      "AUC": roc_auc_score(s["y"], s["p"]) if s["y"].nunique() > 1 else np.nan,
                      "Exactitud balanceada": balanced_accuracy_score(s["y"], yp),
                      "MCC": matthews_corrcoef(s["y"], yp)})
        limpiar()
    return pd.DataFrame(filas)


if EJECUTAR_CV_CNN:
    df_cv_cnn = evaluar_cnn_cv(
        construir_modelo_cnn, X_img, y_img, subject_ids_img,
        callbacks_factory=lambda: [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                                                      restore_best_weights=True)],
        limpiar=tf.keras.backend.clear_session)
    display(df_cv_cnn.round(3))
    print(df_cv_cnn[["AUC", "Exactitud balanceada", "MCC"]].agg(["mean", "std"]).round(3))
    df_cv_cnn.to_csv(CARPETA_SALIDA + "cnn_validacion_cruzada.csv", index=False, encoding="utf-8-sig")

# **Sección B — Benchmarking de 5 Clasificadores Clásicos sobre Métricas Tabulares**

## **B.1. Carga de features (reutiliza la salida del Archivo 1)**
En vez de volver a parsear los `*_tract_stat*.txt` desde cero, esta sección carga directamente
`features_por_sujeto.csv` — el artefacto que el Archivo 1 (Sección 5.2) ya exporta — para evitar
mantener dos implementaciones distintas del mismo parseo. Si ese archivo no existe todavía, se
genera el mismo dataset sintético correlacionado que usa el Archivo 1, para que esta sección también
sea ejecutable de forma independiente.

In [ ]:
def generar_features_sinteticas(n_sujetos=200, semilla=42):
    """Mismo generador sintético correlacionado que el Archivo 1 (Sección 2.4):
    señal conocida y controlada por grupo, nunca etiquetas desconectadas de los
    datos. Devuelve un DataFrame ya en formato ancho (una fila por sujeto).
    """
    rng = np.random.default_rng(semilla)
    tractos = ["arc", "unc", "mdlf", "ilf"]
    lados = ["izquierdo", "derecho"]
    filas = []
    for i in range(1, n_sujetos + 1):
        grupo = "control" if i <= n_sujetos // 2 else "epilepsia"
        fila = {"subject_id": f"S{i:03d}", "grupo_dx": grupo}
        for tracto in tractos:
            for lado in lados:
                media_fa = 0.45 if grupo == "control" else 0.38
                fila[f"{tracto}_{lado}_fa_mean"] = rng.normal(media_fa, 0.05)
                fila[f"{tracto}_{lado}_md_mean"] = rng.normal(
                    0.0008 if grupo == "control" else 0.0009, 0.0001)
                fila[f"{tracto}_{lado}_volume_mm3"] = rng.normal(1500, 200)
        filas.append(fila)
    return pd.DataFrame(filas)


if "df_cohorte_ok" in globals():
    # Etiqueta y covariables SOLO del manifiesto; las métricas vienen de las salidas del Archivo 2
    df_features = df_cohorte_ok[["subject_id", "grupo_dx"]].merge(df_geom_wide, on="subject_id", how="left")
    # Métricas escalares del Archivo 1 (FA, MD, volumen): opcionales, se unen por subject_id
    if os.path.exists(RUTA_FEATURES_TABULARES):
        f1 = pd.read_csv(RUTA_FEATURES_TABULARES)
        f1["_clave"] = f1["subject_id"].map(normalizar_id)
        df_features["_clave"] = df_features["subject_id"].map(normalizar_id)
        if "grupo_dx" in f1.columns:
            comp = df_features[["_clave", "grupo_dx"]].merge(f1[["_clave", "grupo_dx"]], on="_clave",
                                                             suffixes=("", "_archivo1"))
            disc = comp[comp["grupo_dx"] != comp["grupo_dx_archivo1"]]
            if len(disc):
                print(f"⚠ {len(disc)} sujetos con grupo distinto entre el manifiesto y el Archivo 1; "
                      f"se usa el manifiesto: {disc['_clave'].tolist()[:10]}")
        f1 = f1.drop(columns=[c for c in ("subject_id", "grupo_dx") if c in f1.columns])
        df_features = df_features.merge(f1, on="_clave", how="left")
        print(f"Archivo 1 unido: {df_features['_clave'].isin(f1['_clave']).sum()} de {len(df_features)} sujetos "
              f"con métricas escalares.")
    else:
        print("No se encontró features_por_sujeto.csv del Archivo 1: se usan solo las métricas geométricas "
              "del Archivo 2.")
    df_features = df_features.drop(columns=["_clave"], errors="ignore")
    todo_nan = df_features.columns[df_features.isna().all()].tolist()
    constantes = [c for c in df_features.columns.difference(["subject_id", "grupo_dx"])
                  if df_features[c].nunique(dropna=True) <= 1]
    df_features = df_features.drop(columns=list(set(todo_nan + constantes)))
    if todo_nan or constantes:
        print(f"Se descartaron {len(set(todo_nan + constantes))} variables vacías o constantes.")
    print(f"Tabla de características: {df_features.shape[0]} sujetos x {df_features.shape[1] - 2} variables "
          f"({df_features['grupo_dx'].value_counts().to_dict()}).")
    if df_features.shape[1] - 2 > df_features.shape[0]:
        print("⚠ Hay más variables que sujetos: la selección de características (RFECV) se evalúa dentro de la "
              "validación cruzada, nunca sobre todo el conjunto.")
elif os.path.exists(RUTA_FEATURES_TABULARES):
    df_features = pd.read_csv(RUTA_FEATURES_TABULARES)
    invalidos = sorted(set(df_features["grupo_dx"].astype(str)) - {"control", "epilepsia"})
    if invalidos:
        raise ValueError(f"grupo_dx con valores no válidos en {RUTA_FEATURES_TABULARES}: {invalidos}")
    print(f"'{RUTA_FEATURES_TABULARES}' cargado: {df_features.shape[0]} sujetos, "
          f"{df_features.shape[1] - 2} features.")
else:
    print(f"No hay cohorte real ni '{RUTA_FEATURES_TABULARES}'; generando dataset SINTÉTICO de "
          f"validación (ver nota metodológica de la portada)...")
    df_features = generar_features_sinteticas()
    print(f"Dataset sintético generado: {df_features.shape[0]} sujetos, "
          f"{df_features.shape[1] - 2} features.")

df_features.head()

## **B.2. Benchmarking con selección recursiva de características (RFECV)**
A diferencia del Archivo 1 (que entrena los 5 modelos directamente sobre todas las features), aquí
se antepone `RFECV` (*Recursive Feature Elimination with Cross-Validation*) dentro del mismo
pipeline: elimina recursivamente las features menos relevantes, evaluando por validación cruzada,
antes de entrenar el clasificador final — útil cuando el número de features (hasta 4 tractos × 2
lados × ~10 métricas = 80) es grande respecto al número de sujetos. Como `RFECV` requiere un
estimador con `coef_` o `feature_importances_`, los modelos que no lo tienen (SVM, Gradient
Boosting) usan un `RandomForestClassifier` auxiliar solo para la selección de features, y luego
entrenan su propio clasificador final sobre las features seleccionadas.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, matthews_corrcoef,
                              classification_report, confusion_matrix, roc_auc_score)

y = df_features["grupo_dx"].map({"control": 0, "epilepsia": 1})
if y.isna().any():
    raise ValueError("grupo_dx con valores no válidos: "
                     f"{sorted(df_features.loc[y.isna(), 'grupo_dx'].astype(str).unique())}")
X = df_features.drop(columns=["subject_id", "grupo_dx"])

if y.nunique() < 2:
    print("Se requieren ambas clases ('control' y 'epilepsia') en 'grupo_dx' para benchmarking. "
          "Encontradas:", y.unique().tolist())
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    MODELOS_CANDIDATOS = {
        "Regresión Logística": LogisticRegression(max_iter=2000, class_weight="balanced",
                                                    random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=150, class_weight="balanced",
                                                  random_state=42),
        "SVM": SVC(probability=True, class_weight="balanced", random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=800,
                                                         random_state=42),
    }

    resultados_benchmarking = []
    pipelines_entrenados = {}   # se reutilizan en la Sección B.3 (evaluación detallada)

    # RFECV necesita un estimador con 'coef_' (modelos lineales) o
    # 'feature_importances_' (modelos de árboles) -- ambos solo existen DESPUÉS
    # de entrenar, por lo que no se puede detectar con hasattr() de antemano;
    # se declara explícitamente qué modelos los tienen de forma nativa.
    MODELOS_CON_IMPORTANCIA_NATIVA = {"Regresión Logística", "Random Forest", "Gradient Boosting"}
    n_folds_rfecv = min(5, y_train.value_counts().min())
    PASO_RFECV = max(1, X_train.shape[1] // 10)   # elimina ≈10 % de las variables por paso (tiempo razonable con ~100 variables)

    print("=" * 85)
    print("ENTRENAMIENTO, RFECV Y BENCHMARKING DE 5 MODELOS")
    print("=" * 85)

    for nombre_modelo, estimador in MODELOS_CANDIDATOS.items():
        print(f"\nEntrenando: {nombre_modelo}...")

        estimador_selector = (estimador if nombre_modelo in MODELOS_CON_IMPORTANCIA_NATIVA
                               else RandomForestClassifier(random_state=42))
        selector_features = RFECV(
            estimator=estimador_selector, step=PASO_RFECV, cv=StratifiedKFold(n_folds_rfecv), n_jobs=-1,
            scoring="roc_auc", min_features_to_select=3,
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("feature_selection", selector_features),
            ("clasificador", estimador),
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        resultados_benchmarking.append({
            "modelo": nombre_modelo,
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "mcc": matthews_corrcoef(y_test, y_pred),
            "auc_roc": roc_auc_score(y_test, y_proba) if y_test.nunique() > 1 else float("nan"),
            "n_features_seleccionadas": int(selector_features.n_features_),
        })
        pipelines_entrenados[nombre_modelo] = pipeline

    df_resultados_benchmarking = (
        pd.DataFrame(resultados_benchmarking).sort_values("auc_roc", ascending=False)
    )
    df_resultados_benchmarking.to_csv(
        CARPETA_SALIDA + "resultados_benchmarking_rfecv.csv", index=False, encoding="utf-8-sig"
    )

df_resultados_benchmarking if y.nunique() > 1 else None


## **B.2b. Validación cruzada repetida (criterio para elegir el modelo)**
La tabla de la Sección B.2 usa una sola partición 75/25: con ≈100 sujetos el conjunto de prueba tiene ≈25 (≈8 controles), y elegir el "mejor" modelo ordenando por el AUC de esa misma partición sobreestima su desempeño. Aquí cada modelo (con su selección RFECV **dentro** de cada pliegue) se evalúa con validación cruzada estratificada repetida y se **elige por el AUC medio**; la exactitud balanceada y el MCC acompañan al AUC porque con 70:30 la exactitud sola no discrimina. El RFECV elimina ≈10 % de las variables por paso para acotar el tiempo de cómputo.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import make_scorer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

N_PLIEGUES_CV, N_REPETICIONES_CV = 5, 2

if y.nunique() > 1:
    min_clase = int(y.value_counts().min())
    n_pl = max(2, min(N_PLIEGUES_CV, min_clase))
    cv_rep = RepeatedStratifiedKFold(n_splits=n_pl, n_repeats=N_REPETICIONES_CV, random_state=42)
    puntuaciones = {"auc": "roc_auc", "bal_acc": "balanced_accuracy", "mcc": make_scorer(matthews_corrcoef)}

    filas_cv = []
    for nombre_modelo, pipeline in pipelines_entrenados.items():
        pipe_cv = clone(pipeline).set_params(feature_selection__n_jobs=1)   # mismo RFECV que en B.2; el paralelismo se deja al nivel externo
        res = cross_validate(pipe_cv, X, y, cv=cv_rep, scoring=puntuaciones, n_jobs=-1, error_score="raise")
        filas_cv.append({
            "modelo": nombre_modelo,
            "AUC (media)": res["test_auc"].mean(), "AUC (desv.)": res["test_auc"].std(),
            "Exactitud balanceada (media)": res["test_bal_acc"].mean(),
            "MCC (media)": res["test_mcc"].mean(),
            "evaluaciones": len(res["test_auc"]),
        })
    df_cv_benchmarking = pd.DataFrame(filas_cv).sort_values("AUC (media)", ascending=False).reset_index(drop=True)
    df_cv_benchmarking = df_cv_benchmarking.rename(columns={"AUC (media)": "auc_cv"})
    df_cv_benchmarking.to_csv(CARPETA_SALIDA + "benchmarking_validacion_cruzada.csv", index=False,
                              encoding="utf-8-sig")
    print(f"Validación cruzada estratificada: {n_pl} pliegues x {N_REPETICIONES_CV} repeticiones "
          f"(n = {len(y)}; clase minoritaria = {min_clase}).")
    display(df_cv_benchmarking.round(3))
    print(f"Modelo seleccionado por AUC de validación cruzada: {df_cv_benchmarking.iloc[0]['modelo']}")

## **B.3. Evaluación detallada por clase (verdaderos/falsos positivos y negativos)**
Reutiliza los **mismos pipelines ya entrenados** en B.2 (con imputación, escalado y selección de
features incluidos) — no se vuelve a entrenar un pipeline distinto que ignore esos pasos.

In [ ]:
if y.nunique() > 1:
    print("=" * 85)
    print("EVALUACIÓN DETALLADA DE CLASE (FALSOS POSITIVOS Y FALSOS NEGATIVOS)")
    print("=" * 85)

    for nombre_modelo, pipeline in pipelines_entrenados.items():
        y_pred = pipeline.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        print(f"\nModelo: {nombre_modelo}")
        print(f"  Verdaderos Negativos (controles bien clasificados): {tn}")
        print(f"  Falsos Positivos (control catalogado como epilepsia): {fp}")
        print(f"  Falsos Negativos (epilepsia catalogado como control): {fn}")
        print(f"  Verdaderos Positivos (epilepsia bien detectada): {tp}")
        print(classification_report(y_test, y_pred,
                                     target_names=["Control (0)", "Epilepsia (1)"],
                                     zero_division=0))
        print("-" * 65)


# **Comparación Final — CNN vs. Machine Learning Clásico**

In [ ]:
filas_comparacion = []
if "auc_cnn" in globals():
    filas_comparacion.append({"enfoque": "CNN sobre TDI (nivel imagen)",
                               "accuracy": accuracy_score(y_test_img, y_pred_cnn),
                               "auc_roc": auc_cnn})
if "df_pred_sujeto" in globals():
    filas_comparacion.append({
        "enfoque": "CNN sobre TDI (nivel sujeto, voto por promedio)",
        "accuracy": accuracy_score(df_pred_sujeto["y_real"], df_pred_sujeto["y_pred"]),
        "auc_roc": (roc_auc_score(df_pred_sujeto["y_real"], df_pred_sujeto["y_proba_media"])
                    if df_pred_sujeto["y_real"].nunique() > 1 else float("nan")),
    })
if "df_resultados_benchmarking" in globals() and y.nunique() > 1:
    mejor_nombre = (df_cv_benchmarking.iloc[0]["modelo"] if "df_cv_benchmarking" in globals()
                    else df_resultados_benchmarking.iloc[0]["modelo"])
    mejor_clasico = df_resultados_benchmarking[df_resultados_benchmarking["modelo"] == mejor_nombre].iloc[0]
    filas_comparacion.append({"enfoque": f"Clásico elegido por validación cruzada: {mejor_clasico['modelo']}",
                               "accuracy": mejor_clasico["accuracy"],
                               "auc_roc": mejor_clasico["auc_roc"]})

df_comparacion_final = pd.DataFrame(filas_comparacion)
df_comparacion_final.to_csv(CARPETA_SALIDA + "comparacion_cnn_vs_clasico.csv", index=False,
                             encoding="utf-8-sig")
df_comparacion_final.round(3)


# **Despliegue — Persistencia de Artefactos**

In [ ]:
modelo_cnn.save(CARPETA_SALIDA + "modelo_cnn_tdi.keras")

referencias_md = """# Referencias Metodológicas — Archivo 3 (Modelado)

## Arquitectura CNN y buenas prácticas de regularización
Ioffe, S., Szegedy, C. (2015). Batch normalization: Accelerating deep network training by
reducing internal covariate shift. ICML.

Srivastava, N. et al. (2014). Dropout: A simple way to prevent neural networks from
overfitting. Journal of Machine Learning Research, 15(1), 1929-1958.

## Selección de características (RFECV)
Guyon, I., Weston, J., Barnhill, S., Vapnik, V. (2002). Gene selection for cancer
classification using support vector machines. Machine Learning, 46, 389-422.

## Validación agrupada por sujeto (prevención de fuga de datos)
Ver Archivo 1, Sección 5.1 -- mismo principio aplicado aquí a nivel de imagen
(StratifiedGroupKFold por subject_id).

## Track Density Imaging (TDI) como entrada de visión por computador
Calamante, F., Tournier, J-D., Jackson, G.D., Connelly, A. (2010). Track-density imaging
(TDI): Super-resolution white matter imaging using whole-brain track-density mapping.
NeuroImage, 53(4), 1233-1243. https://doi.org/10.1016/j.neuroimage.2010.07.024
"""
with open(CARPETA_SALIDA + "referencias_archivo3.md", "w", encoding="utf-8") as f:
    f.write(referencias_md)

print(f"Artefactos guardados en {CARPETA_SALIDA}:")
print(" - modelo_cnn_tdi.keras                    (Sección A.5-A.6)")
print(" - cnn_curvas_entrenamiento.png, cnn_matriz_confusion.png   (Sección A.6)")
print(" - resultados_benchmarking_rfecv.csv        (Sección B.2)")
print(" - comparacion_cnn_vs_clasico.csv           (Comparación final)")
print(" - manifiesto_cohorte.csv / cohorte_validacion.csv / features_geometricas_cohorte.csv  (Sección 0.2)")
print(" - cnn_metricas_ic95_sujetos.csv, benchmarking_validacion_cruzada.csv  (Secciones A.7b, B.2b)")
print(" - referencias_archivo3.md")


# **Conclusión**
- Se corrigieron dos errores reales de una versión previa de este notebook: (1) la variable
  `modelos` referenciada en el bloque de evaluación detallada nunca estaba definida (usaba
  `MODELOS_CANDIDATOS`) — causaba `NameError`; (2) el bucle principal de benchmarking calculaba
  `accuracy`/`auc_roc` por modelo pero nunca los agregaba a `resultados_benchmarking`, que quedaba
  vacío — ambos corregidos en la Sección B.2-B.3, que ahora reutiliza los mismos pipelines ya
  entrenados (con imputación y escalado) para la evaluación detallada, en vez de reentrenar un
  pipeline distinto e inconsistente.
- Se eliminaron los bloques que entrenaban con **etiquetas generadas aleatoriamente** — cualquier
  resultado de esa versión no tiene validez clínica, sin importar el *accuracy* reportado.
- Se construyó un pipeline de **CNN sobre imágenes TDI reales** (Sección A), con partición
  agrupada por sujeto (igual principio anti-fuga que el Archivo 1) y agregación a nivel de paciente
  por voto de probabilidad promedio — no solo la arquitectura del modelo, sino su conexión real a
  los datos que exporta el Archivo 2.
- El **benchmarking de 5 clasificadores clásicos** (Sección B) ahora reutiliza directamente
  `features_por_sujeto.csv` del Archivo 1 en vez de duplicar su lógica de parseo, añadiendo RFECV
  como paso de selección de características.
- **Siguiente paso recomendado:** correr el Archivo 2 para cada sujeto real de la cohorte,
  organizando sus salidas TDI según la convención de carpetas de la Sección A.3, y el Archivo 1
  sobre el conjunto completo de `*_tract_stat*.txt` — este notebook detecta automáticamente esos
  datos reales y deja de usar los conjuntos sintéticos de validación.
